# 🔍 Prueba de Inferencia Local - Modelo VGG16
## Clasificación de Piezas Industriales

Este notebook te permite probar el modelo entrenado localmente antes de desplegarlo en AWS.

## 📦 Paso 1: Importar Librerías

In [ ]:
import numpy as np
import json
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import matplotlib.pyplot as plt
import os

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras version: {keras.__version__}")

## 🎯 Paso 2: Cargar el Modelo y Metadatos

In [ ]:
# Rutas de los archivos (ajusta según donde tengas el modelo)
MODEL_PATH = '../../vgg16_industrial_classifier_20251014_203947.keras'
CLASSES_PATH = '../../vgg16_industrial_classifier_20251014_203947_classes.json'
METADATA_PATH = '../../vgg16_industrial_classifier_20251014_203947_metadata.json'

# Cargar el modelo
print("Cargando modelo...")
model = keras.models.load_model(MODEL_PATH)
print("✅ Modelo cargado exitosamente")

# Cargar las clases
with open(CLASSES_PATH, 'r') as f:
    classes_data = json.load(f)
    idx_to_class = classes_data['idx_to_class']
    
print(f"\n📋 Clases disponibles ({len(idx_to_class)}):")
for idx, class_name in idx_to_class.items():
    print(f"  {idx}: {class_name}")

In [ ]:
# Ver información del modelo
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)
    
print("\n📊 Información del Modelo:")
print(f"  Arquitectura: {metadata['model_info']['architecture']}")
print(f"  Accuracy en Test: {metadata['performance']['test']['accuracy']:.2%}")
print(f"  Parámetros entrenables: {metadata['architecture']['trainable_params']:,}")
print(f"  Listo para producción: {metadata['analysis']['production_ready']}")

## 🖼️ Paso 3: Función de Preprocesamiento

In [ ]:
def preprocess_image(image_path, target_size=(224, 224)):
    """
    Preprocesa una imagen para el modelo VGG16
    
    Args:
        image_path: Ruta de la imagen
        target_size: Tamaño objetivo (224x224 para VGG16)
    
    Returns:
        Imagen preprocesada lista para predicción
    """
    # Cargar imagen
    img = Image.open(image_path)
    
    # Convertir a RGB si es necesario
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Redimensionar
    img = img.resize(target_size)
    
    # Convertir a array
    img_array = np.array(img)
    
    # Expandir dimensiones (batch)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Normalizar (VGG16 usa normalización específica)
    img_array = tf.keras.applications.vgg16.preprocess_input(img_array)
    
    return img_array, img

print("✅ Función de preprocesamiento lista")

## 🎯 Paso 4: Función de Predicción

In [ ]:
def predict_image(image_path, threshold=0.7):
    """
    Predice la clase de una imagen
    
    Args:
        image_path: Ruta de la imagen
        threshold: Umbral de confianza (default: 0.7)
    
    Returns:
        Diccionario con predicción y probabilidades
    """
    # Preprocesar imagen
    img_array, original_img = preprocess_image(image_path)
    
    # Hacer predicción
    predictions = model.predict(img_array, verbose=0)
    
    # Obtener clase predicha y probabilidad
    predicted_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_idx]
    predicted_class = idx_to_class[str(predicted_idx)]
    
    # Determinar si es identificada o no
    is_identified = confidence >= threshold
    
    # Top 3 predicciones
    top_3_idx = np.argsort(predictions[0])[-3:][::-1]
    top_3 = [(idx_to_class[str(i)], predictions[0][i]) for i in top_3_idx]
    
    return {
        'predicted_class': predicted_class,
        'confidence': float(confidence),
        'is_identified': is_identified,
        'top_3': top_3,
        'all_probabilities': predictions[0].tolist(),
        'original_image': original_img
    }

print("✅ Función de predicción lista")

## 🧪 Paso 5: Probar con una Imagen

In [ ]:
# Ruta al dataset (ajusta según tu estructura)
DATASET_PATH = '../../datos/industrial_classification_data_set'

# Listar las carpetas disponibles
if os.path.exists(DATASET_PATH):
    categories = [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))]
    print(f"📁 Categorías encontradas: {len(categories)}")
    for cat in categories:
        print(f"  - {cat}")
else:
    print("⚠️ Dataset no encontrado. Ajusta la ruta DATASET_PATH")

In [ ]:
# Seleccionar una imagen de prueba
# Cambia esta ruta por una imagen real de tu dataset
test_category = 'screw'  # Cambia por la categoría que quieras probar
test_image_folder = os.path.join(DATASET_PATH, test_category)

if os.path.exists(test_image_folder):
    # Obtener la primera imagen
    images = [f for f in os.listdir(test_image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if images:
        test_image_path = os.path.join(test_image_folder, images[0])
        print(f"🖼️ Imagen de prueba: {test_image_path}")
    else:
        print("⚠️ No se encontraron imágenes en la carpeta")
else:
    print(f"⚠️ Carpeta no encontrada: {test_image_folder}")
    print("\n💡 Ajusta 'test_category' con una de las categorías listadas arriba")

In [ ]:
# Hacer predicción
result = predict_image(test_image_path, threshold=0.7)

# Mostrar resultados
print("\n" + "="*60)
print("🎯 RESULTADO DE LA PREDICCIÓN")
print("="*60)
print(f"\n📸 Imagen: {os.path.basename(test_image_path)}")
print(f"📁 Categoría real: {test_category}")
print(f"\n🤖 Predicción: {result['predicted_class']}")
print(f"📊 Confianza: {result['confidence']:.2%}")
print(f"✅ Identificada: {'SÍ' if result['is_identified'] else 'NO (baja confianza)'}")

print(f"\n🏆 Top 3 Predicciones:")
for i, (class_name, prob) in enumerate(result['top_3'], 1):
    print(f"  {i}. {class_name}: {prob:.2%}")

# Visualizar imagen
plt.figure(figsize=(8, 6))
plt.imshow(result['original_image'])
plt.axis('off')
plt.title(f"Predicción: {result['predicted_class']} ({result['confidence']:.2%})\nReal: {test_category}", 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Verificar si la predicción es correcta
is_correct = result['predicted_class'] == test_category
print(f"\n{'✅ ¡CORRECTO!' if is_correct else '❌ INCORRECTO'}")

## 🔄 Paso 6: Probar con Múltiples Imágenes

In [ ]:
def test_multiple_images(category, num_images=5, threshold=0.7):
    """
    Prueba el modelo con múltiples imágenes de una categoría
    """
    category_path = os.path.join(DATASET_PATH, category)
    
    if not os.path.exists(category_path):
        print(f"⚠️ Categoría no encontrada: {category}")
        return
    
    images = [f for f in os.listdir(category_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    images = images[:num_images]
    
    results = []
    
    print(f"\n🧪 Probando {len(images)} imágenes de '{category}'...\n")
    
    for img_name in images:
        img_path = os.path.join(category_path, img_name)
        result = predict_image(img_path, threshold)
        
        is_correct = result['predicted_class'] == category
        results.append(is_correct)
        
        status = "✅" if is_correct else "❌"
        identified = "✓" if result['is_identified'] else "✗"
        
        print(f"{status} {img_name[:30]:30s} | Pred: {result['predicted_class']:25s} | Conf: {result['confidence']:.2%} | ID: {identified}")
    
    accuracy = sum(results) / len(results) * 100
    print(f"\n📊 Accuracy en esta muestra: {accuracy:.1f}% ({sum(results)}/{len(results)})")
    
    return results

# Probar con una categoría
test_multiple_images('screw', num_images=10)

## 📊 Paso 7: Análisis de Confianza

In [ ]:
# Probar diferentes umbrales
def analyze_thresholds(category, num_images=20):
    """
    Analiza cómo diferentes umbrales afectan la clasificación
    """
    category_path = os.path.join(DATASET_PATH, category)
    images = [f for f in os.listdir(category_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    images = images[:num_images]
    
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
    results = {t: {'correct': 0, 'identified': 0, 'total': 0} for t in thresholds}
    
    for img_name in images:
        img_path = os.path.join(category_path, img_name)
        result = predict_image(img_path, threshold=0.5)  # Usar umbral bajo para obtener todas las predicciones
        
        for threshold in thresholds:
            is_identified = result['confidence'] >= threshold
            is_correct = result['predicted_class'] == category
            
            results[threshold]['total'] += 1
            if is_identified:
                results[threshold]['identified'] += 1
                if is_correct:
                    results[threshold]['correct'] += 1
    
    print(f"\n📊 Análisis de Umbrales para '{category}' ({num_images} imágenes)\n")
    print(f"{'Umbral':<10} {'Identificadas':<15} {'Correctas':<15} {'Accuracy':<15} {'% Identificadas'}")
    print("-" * 70)
    
    for threshold in thresholds:
        r = results[threshold]
        accuracy = (r['correct'] / r['identified'] * 100) if r['identified'] > 0 else 0
        pct_identified = r['identified'] / r['total'] * 100
        
        print(f"{threshold:<10.1f} {r['identified']:<15} {r['correct']:<15} {accuracy:<15.1f} {pct_identified:.1f}%")

# Analizar
analyze_thresholds('screw', num_images=20)

## 🎯 Paso 8: Simular el Flujo de AWS

Esta función simula lo que hará Lambda en AWS

In [ ]:
def simulate_aws_classification(image_path, threshold=0.7):
    """
    Simula el flujo completo que se ejecutará en AWS:
    1. Imagen llega a S3
    2. Lambda invoca SageMaker endpoint
    3. Modelo clasifica
    4. Lambda mueve imagen a carpeta correspondiente
    """
    print("\n" + "="*60)
    print("🔄 SIMULACIÓN DE FLUJO AWS")
    print("="*60)
    
    # Paso 1: Imagen llega a S3
    print(f"\n1️⃣ Imagen subida a S3: {os.path.basename(image_path)}")
    
    # Paso 2: Lambda se dispara
    print("2️⃣ Lambda detecta nuevo archivo, invocando SageMaker...")
    
    # Paso 3: SageMaker hace predicción
    result = predict_image(image_path, threshold)
    print(f"3️⃣ SageMaker responde: {result['predicted_class']} ({result['confidence']:.2%})")
    
    # Paso 4: Lambda decide dónde mover
    if result['is_identified']:
        destination = f"s3://bucket-industrial/clasificadas/{result['predicted_class']}/"
        print(f"4️⃣ ✅ Confianza alta - Moviendo a: {destination}")
    else:
        destination = "s3://bucket-industrial/no-identificadas/"
        print(f"4️⃣ ⚠️ Confianza baja - Moviendo a: {destination}")
    
    print("\n✅ Proceso completado")
    
    return {
        'source': image_path,
        'destination': destination,
        'prediction': result
    }

# Probar simulación
simulate_aws_classification(test_image_path)

## ✅ Resumen

En este notebook has:

1. ✅ Cargado el modelo VGG16 entrenado
2. ✅ Probado predicciones con imágenes individuales
3. ✅ Evaluado el modelo con múltiples imágenes
4. ✅ Analizado diferentes umbrales de confianza
5. ✅ Simulado el flujo completo de AWS

### 🚀 Próximos Pasos:

1. Preparar el código de inferencia para SageMaker
2. Crear la función Lambda
3. Desplegar en AWS

### 💡 Recomendaciones:

- **Umbral óptimo**: Basado en tus pruebas, elige un umbral entre 0.6-0.8
- **Carpeta no-identificadas**: Revisa periódicamente para reentrenar el modelo
- **Monitoreo**: En producción, registra todas las predicciones para análisis